In [23]:
from dataclasses import dataclass, field
from typing import Optional

from kloppy.domain import Event, EventType


@dataclass
class State:
    total_time: float
    in_play_time: float
    in_play: bool
    prev_event: Optional[Event] = field(repr=False, default=None)


def is_ball_alive(event: Event) -> bool:
    return (
        event.event_type in (EventType.PASS, EventType.SHOT)
        
        # This is Statsbomb specific and not standardized by kloppy
        or event.event_name == 'Half Start'
    )

def is_ball_dead(event: Event) -> bool:
    return (
        event.event_type in (EventType.BALL_OUT, EventType.FOUL_COMMITTED) 
        
        # This is Statsbomb specific and not standardized by kloppy
        or event.event_name == 'Half End'
    )

def reduce(state: State, event: Event) -> State:
    if is_ball_alive(event):
        in_play = True
    elif is_ball_dead(event):
        in_play = False
    else:
        in_play = state.in_play
    
    time_increase = (
        event.timestamp - state.prev_event.timestamp 
        if state.prev_event and state.prev_event.period == event.period else 
        event.timestamp
    )
    
    return State(
        total_time=state.total_time + time_increase,
        in_play_time=(
            state.in_play_time + time_increase 
            if state.in_play else 
            state.in_play_time
        ),
        in_play=in_play,
        prev_event=event
    )
            
        

In [36]:
from kloppy.utils import performance_logging

def calculate_playing_time(dataset):
    with performance_logging("calculate playing time"):
        state = State(total_time=0.0, in_play_time=0.0, in_play=False)

        for event in dataset:
            state = reduce(state, event)
#             print(f"Event: {event.event_name} {event.timestamp} - {state}")

    match_name = f"{dataset.metadata.teams[0]} - {dataset.metadata.teams[1]}"
    print(f"{match_name}\n{state}\n")


In [37]:
from kloppy import statsbomb, wyscout

dataset = wyscout.load_open_data()
calculate_playing_time(dataset)

dataset = statsbomb.load_open_data()
calculate_playing_time(dataset)

calculate playing time took: 1.47ms 
Huddersfield Town FC - Manchester City FC
State(total_time=5758.163138, in_play_time=4015.1247099999996, in_play=True)

calculate playing time took: 4.09ms 
Barcelona - Deportivo Alavés
State(total_time=5557.319999999992, in_play_time=4576.46799999999, in_play=False)

